# 🏆 Lichess Titled Players Beaten Tracker

Track how many titled players you've beaten on Lichess!

## 🚀 How to Use
1. **Run all cells** (Cell → Run All or Shift+Enter on each cell)
2. **Enter your Lichess username** in the form below
3. **Select time control** (All, Bullet, Blitz, etc.)
4. **Choose game scope** (start with 500 games)
5. **Click "Analyze Games"** and wait for results
6. **Download CSV** for detailed game data

## ✨ Features
- 🎯 **Counts titled players beaten**: GM, IM, FM, CM, WGM, WIM, WFM, WCM, NM, LM
- 📊 **Clean summary**: Just the numbers you want to see
- 💾 **Auto-export CSV**: Detailed results saved automatically
- 🆓 **Completely free**: No registration or tokens required

In [4]:
# Install and import required packages
import subprocess
import sys

def install_package(package):
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package], 
                            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    except:
        print(f"⚠️ Failed to install {package}")

# Check and install required packages
required_packages = ['requests', 'pandas', 'ipywidgets']
for package in required_packages:
    try:
        __import__(package)
    except ImportError:
        print(f"📦 Installing {package}...")
        install_package(package)

# Import all required modules
import requests
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import json
from datetime import datetime
import time

print("✅ All packages loaded successfully!")

✅ All packages loaded successfully!


In [5]:
class LichessTitledPlayersTracker:
    """Track titled players beaten on Lichess"""
    
    def __init__(self):
        self.base_url = "https://lichess.org/api"
        self.session = requests.Session()
        
        # Set up headers for free API access
        self.session.headers.update({
            'User-Agent': 'Lichess Titled Players Tracker v1.0',
            'Accept': 'application/x-ndjson'
        })
        
        # All possible titles on Lichess
        self.titles = ['GM', 'IM', 'FM', 'CM', 'WGM', 'WIM', 'WFM', 'WCM', 'NM', 'LM']
        
        # Time control mappings
        self.time_controls = {
            'bullet': 'bullet',
            'blitz': 'blitz', 
            'rapid': 'rapid',
            'classical': 'classical',
            'correspondence': 'correspondence'
        }
    
    def get_user_games(self, username, time_control=None, max_games=None):
        """Fetch games for a user from Lichess API"""
        url = f"{self.base_url}/games/user/{username}"
        
        params = {
            'rated': 'true',
            'format': 'json'
        }
        
        if max_games:
            params['max'] = max_games
        
        if time_control and time_control != 'all':
            params['perfType'] = time_control
        
        try:
            scope_text = f"last {max_games}" if max_games else "all"
            tc_text = time_control if time_control != 'all' else "all time controls"
            print(f"🔍 Fetching {scope_text} {tc_text} games for {username}...")
            
            response = self.session.get(url, params=params, stream=True, timeout=30)
            response.raise_for_status()
            
            games = []
            for line in response.iter_lines():
                if line:
                    try:
                        game = json.loads(line.decode('utf-8'))
                        games.append(game)
                    except json.JSONDecodeError:
                        continue
            
            print(f"✅ Fetched {len(games)} games")
            return games
            
        except requests.exceptions.RequestException as e:
            print(f"❌ Error fetching games: {e}")
            return []
    
    def analyze_titled_opponents(self, username, games):
        """Analyze games to find titled opponents that were beaten"""
        titled_beaten = {title: [] for title in self.titles}
        total_games_analyzed = 0
        wins_against_titled = 0
        
        for game in games:
            total_games_analyzed += 1
            
            # Get player data
            players = game.get('players', {})
            white_player = players.get('white', {})
            black_player = players.get('black', {})
            
            # Determine user's color and opponent
            white_name = white_player.get('user', {}).get('name', '').lower()
            black_name = black_player.get('user', {}).get('name', '').lower()
            
            if white_name == username.lower():
                user_color = 'white'
                opponent_data = black_player
            elif black_name == username.lower():
                user_color = 'black'
                opponent_data = white_player
            else:
                continue
            
            # Check if user won
            winner = game.get('winner')
            if winner != user_color:
                continue
            
            # Check if opponent has a title
            opponent_user = opponent_data.get('user', {})
            opponent_title = opponent_user.get('title')
            opponent_username = opponent_user.get('name', 'Unknown')
            
            if opponent_title and opponent_title in self.titles:
                wins_against_titled += 1
                
                # Get time control (perf field is a string like 'bullet', 'blitz')
                time_control_name = game.get('perf', 'Unknown')
                
                game_info = {
                    'opponent': opponent_username,
                    'date': datetime.fromtimestamp(game.get('createdAt', 0) / 1000).strftime('%Y-%m-%d'),
                    'time_control': time_control_name.title() if time_control_name else 'Unknown',
                    'rating': opponent_data.get('rating', 'N/A'),
                    'game_url': f"https://lichess.org/{game.get('id', '')}"
                }
                titled_beaten[opponent_title].append(game_info)
        
        print(f"📊 Analyzed {total_games_analyzed} games, found {wins_against_titled} wins against titled players")
        return titled_beaten

print("🚀 Tracker class loaded successfully!")

🚀 Tracker class loaded successfully!


In [6]:
# 🎮 MAIN APPLICATION - Clean Interface

# Username input
username_input = widgets.Text(
    value='',
    placeholder='Enter your Lichess username',
    description='Username:',
    style={'description_width': 'initial'},
    layout={'width': '300px'}
)

# Time control selection
time_control_dropdown = widgets.Dropdown(
    options=[
        ('All time controls', 'all'),
        ('Bullet', 'bullet'),
        ('Blitz', 'blitz'),
        ('Rapid', 'rapid'),
        ('Classical', 'classical'),
        ('Correspondence', 'correspondence')
    ],
    value='all',
    description='Time Control:',
    style={'description_width': 'initial'}
)

# Game scope selection
game_scope_dropdown = widgets.Dropdown(
    options=[
        ('All games (complete history)', 'all'),
        ('Recent 100 games (quick test)', 100),
        ('Recent 500 games', 500),
        ('Recent 1000 games', 1000),
        ('Recent 2000 games', 2000),
        ('Recent 5000 games', 5000)
    ],
    value=500,
    description='Game Scope:',
    style={'description_width': 'initial'}
)

# Analyze button
analyze_button = widgets.Button(
    description='🔍 Analyze Games',
    button_style='primary',
    icon='search',
    layout={'width': '200px', 'height': '40px'}
)

# Output area
output_area = widgets.Output()

# Create tracker instance
tracker = None

def display_clean_results(titled_beaten, username):
    """Display clean summary and auto-export CSV"""
    print("\n" + "=" * 50)
    print(f"🏆 {username.upper()} vs TITLED PLAYERS")
    print("=" * 50)
    
    total_titled_beaten = 0
    results_data = []
    
    # Prepare data and count
    for title in tracker.titles:
        count = len(titled_beaten[title])
        total_titled_beaten += count
        
        # Add to CSV data
        for game_info in titled_beaten[title]:
            results_data.append({
                'Title': title,
                'Opponent': game_info['opponent'],
                'Rating': game_info['rating'],
                'Date': game_info['date'],
                'Time_Control': game_info['time_control'],
                'Game_URL': game_info['game_url']
            })
    
    if total_titled_beaten > 0:
        # Clean summary display
        print("\n📊 RESULTS:")
        for title in tracker.titles:
            count = len(titled_beaten[title])
            if count > 0:
                print(f"  {title}: {count}")
        
        print(f"\n🎉 TOTAL: {total_titled_beaten} titled players beaten!")
        
        # Auto-export to CSV
        if results_data:
            df = pd.DataFrame(results_data)
            filename = f"{username}_titled_players.csv"
            df.to_csv(filename, index=False)
            print(f"\n💾 Results saved to: {filename}")
            print(f"📁 Contains {len(results_data)} games with full details")
            
            # Show just a preview
            print("\n📋 Preview (first 3 games):")
            preview_df = df[['Title', 'Opponent', 'Rating', 'Date', 'Time_Control']].head(3)
            display(preview_df)
    else:
        print("\n😔 No titled players beaten in analyzed games")
        print("💡 Try: larger game scope, different time control")
        print("🎯 Keep playing - beating titled players is rare!")

def analyze_games(button):
    """Main analysis function"""
    global tracker
    
    with output_area:
        clear_output()
        
        # Validate username
        username = username_input.value.strip()
        if not username:
            print("❌ Please enter a username")
            return
        
        # Initialize tracker
        tracker = LichessTitledPlayersTracker()
        
        # Get parameters
        time_control = time_control_dropdown.value
        game_scope = game_scope_dropdown.value
        
        # Display analysis info
        scope_text = "ALL games" if game_scope == 'all' else f"recent {game_scope} games"
        print(f"🎯 Analyzing {scope_text} for {username}")
        print(f"⏱️ Time control: {time_control_dropdown.label}")
        print("-" * 50)
        
        # Fetch games
        max_games = None if game_scope == 'all' else game_scope
        games = tracker.get_user_games(username, time_control if time_control != 'all' else None, max_games)
        
        if not games:
            print("❌ No games found")
            print("💡 Check username spelling and try again")
            return
        
        # Analyze for titled opponents
        print("\n🎯 Analyzing for titled opponents...")
        titled_beaten = tracker.analyze_titled_opponents(username, games)
        
        # Display clean results
        display_clean_results(titled_beaten, username)

# Connect button
analyze_button.on_click(analyze_games)

# Display UI
display(HTML("<h2>🏆 Lichess Titled Players Tracker</h2>"))
display(HTML("<p><strong>Enter your details and click analyze!</strong></p>"))

display(widgets.VBox([
    username_input,
    time_control_dropdown,
    game_scope_dropdown,
    widgets.HTML('<br>'),
    analyze_button,
    widgets.HTML('<br>'),
    output_area
]))

print("\n🎮 Ready! Fill in your details above and click 'Analyze Games'")
print("📊 Results will show clean counts + auto-save detailed CSV")


🎮 Ready! Fill in your details above and click 'Analyze Games'
📊 Results will show clean counts + auto-save detailed CSV
